In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS oil_stock")
spark.sql("USE CATALOG oil_stock")

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_df = spark.table("oil_stock.silver.market_5m")

In [0]:
w = Window.partitionBy("symbol").orderBy("timestamp_utc")

returns_df = (
    silver_df
    .withColumn(
        'previous_close',
        F.lag('close').over(w)
    )
    .withColumn(
        'return_5m_pct',
        (F.col('close') / F.col('previous_close')) - 1
    )
)
display(returns_df)


In [0]:
gold_df = (
    returns_df
    .groupBy('timestamp_utc')
    .agg(
        F.max(
            F.when(F.col('symbol') == 'EQNR.OL', F.col('close'))
        ).alias('eqnr_price'),
        F.max(
            F.when(F.col('symbol') == 'BZ=F', F.col('close'))
        ).alias('brent_price'),
        F.max(
            F.when(F.col('symbol') == 'EQNR.OL', F.col('return_5m_pct'))
        ).alias('eqnr_return_5m_pct'),
        F.max(
            F.when(F.col('symbol') == 'BZ=F', F.col('return_5m_pct'))
        ).alias('brent_return_5m_pct')
    )
    .filter(
        F.col('eqnr_price').isNotNull() &
        F.col('brent_price').isNotNull()
    )
    .withColumn(
        'timestamp_oslo',
        F.from_utc_timestamp('timestamp_utc', 'Europe/Oslo')
    )
    .withColumn(
        "trading_date_oslo",
        F.to_date(F.col("timestamp_oslo"))
    )
    .orderBy('timestamp_utc')
)

display(gold_df)


In [0]:
gold_df.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable('oil_stock.gold.market_5m')